In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [37]:
dfVendas = pd.read_csv('dfVendas.csv')
produtos = pd.read_csv('produtos.csv')

In [38]:
dfVendas.head()

,ID_Venda,ID_Cliente,ID_Produto,Data,Quantidade,Canal,Valor_Total,ID_Campanha
0,1,350,45,2022-09-23,4,Online,5795.64,17
1,2,925,68,2024-02-09,1,Loja Física,4614.41,16
2,3,915,75,2022-09-11,4,Online,2057.48,17
3,4,494,47,2024-05-10,4,Loja Física,12864.44,9
4,5,523,79,2023-10-15,4,Loja Física,13317.28,5


In [39]:
desempenho_produtos = dfVendas.groupby('ID_Produto')['Valor_Total'].sum().reset_index()
desempenho_produtos = desempenho_produtos.sort_values(by='Valor_Total', ascending=False)
desempenho_produtos.head()

,ID_Produto,Valor_Total
86,87,2842287.90
28,29,2841430.92
60,61,2793744.31
15,16,2779379.13
76,77,2716313.55


In [7]:
desempenho_total = desempenho_produtos.merge(produtos, on='ID_Produto') #merge = adiconar tabela produtos (marca, nomed do produto)
desempenho_total.head()

,ID_Produto,Valor_Total,Nome_Produto,Categoria,Preco,Marca
0,87,2842287.90,Liquidificador Oster,Móveis,4986.47,Oster
1,29,2841430.92,Câmera de Segurança Intelbras,Eletrônicos,4907.48,Intelbras
2,61,2793744.31,Memória RAM Corsair 16GB,Informática,4808.51,Corsair
3,16,2779379.13,Teclado Logitech,Eletrônicos,4954.33,Logitech
4,77,2716313.55,Smartwatch Apple Watch,Eletrodomésticos,4416.77,Apple


In [8]:
TotalVendasProduto = desempenho_total.groupby(['Nome_Produto', 'Categoria'])['Valor_Total'].sum().reset_index()
TotalVendasProduto = TotalVendasProduto.sort_values(by="Valor_Total", ascending=False)
TotalVendasProduto.head()

,Nome_Produto,Categoria,Valor_Total
46,Liquidificador Oster,Móveis,2842287.90
21,Câmera de Segurança Intelbras,Eletrônicos,2841430.92
49,Memória RAM Corsair 16GB,Informática,2793744.31
91,Teclado Logitech,Eletrônicos,2779379.13
83,Smartwatch Apple Watch,Eletrodomésticos,2716313.55


In [11]:
media = np.mean(TotalVendasProduto['Valor_Total'])  # media = média aritmética
mediana = np.median(TotalVendasProduto['Valor_Total'])  # mediana = valor central
q1 = np.percentile(TotalVendasProduto['Valor_Total'], 25)  # q1 = primeiro quartil
q2 = np.percentile(TotalVendasProduto['Valor_Total'], 50)  # q2 = segundo quartil = mediana
q3 = np.percentile(TotalVendasProduto['Valor_Total'], 75)  # q3 = terceiro quartil
distancia = (media - mediana) / mediana  # distancia = distância entre média e mediana
iqr = q3 - q1  # iqr = intervalo interquartil
limite_inferior = q1 - 1.5 * iqr  # limite_inferior = limite inferior para detectar outliers
limite_superior = q3 + 1.5 * iqr  # limite_superior = limite superior para detectar outliers
outliers = TotalVendasProduto['Valor_Total'][(TotalVendasProduto['Valor_Total'] < limite_inferior) | (TotalVendasProduto['Valor_Total'] > limite_superior)]  # outliers = valores que estão fora do limite inferior e superior
variancia = TotalVendasProduto["Valor_Total"].var()  # variancia = medida de dispersão dos dados
desvio_padrao = TotalVendasProduto["Valor_Total"].std()  # desvio_padrao = raiz quadrada da variância
coeficiente_variacao = (desvio_padrao / media) * 100  # coeficiente_variacao = medida relativa da dispersão
distancia_variancia = (variancia / (media ** 2)) * 100  # distancia_variancia = relação entre variância e o quadrado da média

print(f"Média: {media:.2f}")
print(f"Mediana: {mediana:.2f}")
print(f"Q1: {q1:.2f}")
print(f"Q2: {q2:.2f}")
print(f"Q3: {q3:.2f}")
print(f"Distância: {distancia * 100:.2f}%")
print(f"Variância: {variancia:.2f}")
print(f"Desvio Padrão: {desvio_padrao:.2f}")
print(f"Coeficiente de Variação: {coeficiente_variacao:.2f}%")
print(f"Distância da Variância em relação à Média: {distancia_variancia:.2f}%")

Média: 1510343.69
Mediana: 1642007.36
Q1: 913144.58
Q2: 1642007.36
Q3: 2075155.80
Distância: -8.02%
Variância: 585307022338.54
Desvio Padrão: 765053.61
Coeficiente de Variação: 50.65%
Distância da Variância em relação à Média: 25.66%


MÉDIA ESTÁ 8,02% ABAIXO DA MEDIANA = ASSIMETRIA NEGATIVA
50% DAS VENDAS ESTÃO ENTRE R$ 913 MIL E R$ 2,07 MILHÕES (Q1-Q3)
PRODUTOS ABAIXO DE Q1 SÃO OS 25% PIORES(Q1)
DESVIO PADRÃO: R$ 765.053,61 = ALTA VARIAÇÃO ENTRE OS PRODUTOS.
COEFICIENTE DE VARIAÇÃO: 50,65% = DISPERSÃO MUITO ALTA EM RELAÇÃO À MÉDIA, INDICANDO HETEROGENEIDADE.
VARIÂNCIA = REFLETE GRANDE DISPERSÃO, REFORÇANDO QUE OS VALORES NÃO SÃO HOMOGÊNEOS.

In [31]:
Outlier_inferior = TotalVendasProduto.loc[TotalVendasProduto["Valor_Total"] <= limite_inferior]
Outlier_inferior

,Nome_Produto,Categoria,Valor_Total


OUTLIER INFERIOR: LIMITE INFERIOR NEGATIVO NÃO HÁ OUTLIERS INFERIORES VÁLIDOS 

In [ ]:
df_produtos_pior_desempenho = TotalVendasProduto[TotalVendasProduto['Valor_Total'] < q1] # Filtra os produtos com Valor_Total menor que Q1 (25% piores)
df_produtos_pior_desempenho= df_produtos_pior_desempenho.sort_values(by='Valor_Total') # Ordenar
df_produtos_pior_desempenho

,Nome_Produto,Categoria,Valor_Total
87,"TV LG 50""",Eletrônicos,78927.61
45,Leitor de Cartão SD,Informática,96129.76
76,Secador de Cabelo Gama,Eletrônicos,139134.19
23,Echo Dot Amazon,Móveis,143869.02
42,Impressora HP Deskjet,Móveis,146272.40
25,Estabilizador APC,Eletrodomésticos,164080.14
22,Drone DJI Mini 2,Informática,188727.00
10,Caixa de Som JBL Flip,Eletrônicos,199619.16
84,Smartwatch Huawei,Informática,215046.76
13,Chromecast Google,Informática,263871.81
